# RICE2 Compact U-Net: reproducible GPU training

This notebook trains the project’s final RICE2 Compact U-Net baseline, evaluates it only on the held-out test set, and saves the checkpoint and reports to Google Drive. RICE2 clear references are temporally nearby observations, so results must be described as **cloud reconstruction**, not same-instant physical truth.

## Before running

In Colab select **Runtime → Change runtime type → T4 GPU**. Upload these two archives to `MyDrive/rice2-prototype/`:

- `project_source.zip`: a fresh source-only archive of this repository, including `src/`, `configs/`, `tests/`, and this notebook.
- `RICE2.zip`: the archive containing `RICE2/cloud`, `RICE2/label`, and `RICE2/mask`.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing.'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/rice2-prototype')
PROJECT_ARCHIVE = DRIVE_ROOT / 'project_source.zip'
RICE2_ARCHIVE = DRIVE_ROOT / 'RICE2.zip'
assert PROJECT_ARCHIVE.is_file(), f'Missing {PROJECT_ARCHIVE}'
assert RICE2_ARCHIVE.is_file(), f'Missing {RICE2_ARCHIVE}'
print('Drive inputs verified:', PROJECT_ARCHIVE.name, RICE2_ARCHIVE.name)

In [ ]:
import os, shutil, zipfile
from pathlib import Path

WORKSPACE = Path('/content/liss4-cloud-removal')
shutil.rmtree(WORKSPACE, ignore_errors=True)
WORKSPACE.mkdir(parents=True)
with zipfile.ZipFile(PROJECT_ARCHIVE) as archive:
    archive.extractall(WORKSPACE)
# Support either a flat archive or an archive with one repository root directory.
if not (WORKSPACE / 'pyproject.toml').is_file():
    candidates = list(WORKSPACE.glob('*/pyproject.toml'))
    assert len(candidates) == 1, 'project_source.zip must contain exactly one project root'
    nested = candidates[0].parent
    for item in nested.iterdir():
        shutil.move(str(item), WORKSPACE / item.name)
    nested.rmdir()
os.chdir(WORKSPACE)
data_root = WORKSPACE / 'data' / 'raw'
data_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(RICE2_ARCHIVE) as archive:
    archive.extractall(WORKSPACE)
assert (data_root / 'RICE2' / 'cloud').is_dir(), 'Expected data/raw/RICE2/cloud after extraction'
print('Workspace prepared:', WORKSPACE)

In [ ]:
!pip -q install -U pip
# Keep Colab's CUDA-enabled PyTorch; install the project without dependency resolution.
!pip -q install mlflow rasterio pyyaml pillow pytest ruff
!pip -q install -e . --no-deps
!python -m cloud_removal.rice2 --dataset-root data/raw/RICE2 --output data/manifests/rice2_manifest.json
!python -m pytest tests/test_rice2.py tests/test_train.py -q
!ruff check src/cloud_removal/rice2.py src/cloud_removal/train_rice2.py src/cloud_removal/evaluate_rice2.py tests/test_rice2.py

In [ ]:
# First validate the entire GPU data-to-checkpoint path.
!python -m cloud_removal.train_rice2 --config configs/train_rice2_compact_unet.yaml --smoke-test

In [ ]:
# Full training: 40 epochs maximum, with early stopping after 8 non-improving validation epochs.
!python -m cloud_removal.train_rice2 --config configs/train_rice2_compact_unet.yaml

In [ ]:
# This is the only final metric report to present: the untouched test split.
!python -m cloud_removal.evaluate_rice2 --checkpoint checkpoints/rice2/best_rice2_compact_unet.pt --manifest data/manifests/rice2_manifest.json --split test --output reports/rice2_test_metrics.json --device cuda

import json
report = json.load(open('reports/rice2_test_metrics.json'))
print(json.dumps(report, indent=2))

In [ ]:
# Persist every deliverable to Drive.
import shutil
OUTPUT = DRIVE_ROOT / 'outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)
for relative in [
    'checkpoints/rice2/best_rice2_compact_unet.pt',
    'checkpoints/rice2/latest_rice2_compact_unet.pt',
    'data/manifests/rice2_manifest.json',
    'reports/rice2_test_metrics.json',
]:
    source = WORKSPACE / relative
    assert source.is_file(), f'Missing expected output: {source}'
    shutil.copy2(source, OUTPUT / source.name)
print('Saved final deliverables to', OUTPUT)